# 01 — Grid Definition (Houston / HCAD)

Defines a regular grid over Houston using HCAD building data and derives the **Y variable** (`zone_type`) from HCAD `landuse`.

**Method:**
1. Load consolidated HCAD CSV
2. Geocode using parcel shapefile (reproject to WGS84)
3. Generate a regular grid covering the HCAD bounding box
4. Assign HCAD records to grid cells via spatial indexing
5. Compute area-weighted `landuse` distribution per cell → derive zone_type

**Output columns:** `cell_id`, `cell_lat`, `cell_lon`, `zone_type`, `cell_lot_count`

**Output file:** `csv/Houston/01_grid_definition.csv`

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.neighbors import BallTree
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

# Load config
with open("grid.json", encoding="utf-8") as f:
    config = json.load(f)

HCAD_PATH = config["hcad_path"]
CELL_SIZE_M = config["grid_cell_size_m"]
MIN_LOTS = config["min_lots_per_cell"]
CSV_DIR = config.get("csv_dir", "csv")

os.makedirs(CSV_DIR, exist_ok=True)

print("01 — Grid Definition (Houston / HCAD)")
print(f"HCAD file:       {HCAD_PATH}")
print(f"Grid cell size:  {CELL_SIZE_M}m")
print(f"Min lots/cell:   {MIN_LOTS}")
print(f"CSV dir:         {CSV_DIR}/")

In [ ]:
# Load HCAD data
df_hcad = pd.read_csv(HCAD_PATH, dtype={"landuse": str})
df_hcad = df_hcad.dropna(subset=["latitude", "longitude"]).copy()

print(f"Loaded {len(df_hcad):,} HCAD records (before geocoding)")
print(f"Initial coordinates (placeholder):")
print(f"  Lat: {df_hcad['latitude'].min():.4f} to {df_hcad['latitude'].max():.4f}")
print(f"  Lon: {df_hcad['longitude'].min():.4f} to {df_hcad['longitude'].max():.4f}")
print(f"  Unique lat/lon pairs: {df_hcad[['latitude', 'longitude']].drop_duplicates().shape[0]:,}")
print(f"\nColumns available: {df_hcad.columns.tolist()}")

In [ ]:
# CRITICAL: Geocoding step — join HCAD to shapefile with proper CRS handling
print("\n" + "="*60)
print("GEOCODING: Joining HCAD to parcel shapefile")
print("="*60)

shp_path = "Houston_data/shp_files/Parcels.shp"
try:
    parcels_gdf = gpd.read_file(shp_path)
    print(f"\n✓ Loaded {len(parcels_gdf):,} parcels from shapefile")
    print(f"  CRS: {parcels_gdf.crs}")
    print(f"  Columns: {parcels_gdf.columns.tolist()[:10]}...")  # Show first 10
    
    # CRITICAL: Reproject to WGS84 BEFORE extracting centroids
    if parcels_gdf.crs != "EPSG:4326":
        print(f"\n  Converting {parcels_gdf.crs} → EPSG:4326 (WGS84)...")
        parcels_gdf = parcels_gdf.to_crs("EPSG:4326")
        print(f"  ✓ Converted successfully")
    
    # Extract centroids in correct projection
    parcels_gdf['latitude'] = parcels_gdf.geometry.centroid.y
    parcels_gdf['longitude'] = parcels_gdf.geometry.centroid.x
    
    print(f"\n  Centroid coordinates after reprojection:")
    print(f"    Lat: {parcels_gdf['latitude'].min():.4f} to {parcels_gdf['latitude'].max():.4f}")
    print(f"    Lon: {parcels_gdf['longitude'].min():.4f} to {parcels_gdf['longitude'].max():.4f}")
    
    # Find and normalize account numbers
    print(f"\n  Normalizing account numbers...")
    print(f"    Sample HCAD acct: {df_hcad['acct'].head().tolist()}")
    print(f"    Sample shapefile HCAD_NUM: {parcels_gdf['HCAD_NUM'].head().tolist()}")
    
    # Try different padding lengths
    best_pad = 11
    best_match = 0
    
    for pad_len in [10, 11, 12]:
        df_hcad[f'acct_pad{pad_len}'] = df_hcad['acct'].astype(str).str.zfill(pad_len)
        parcels_gdf[f'hcad_pad{pad_len}'] = parcels_gdf['HCAD_NUM'].astype(str).str.zfill(pad_len)
        
        # Count potential matches
        matches = df_hcad[f'acct_pad{pad_len}'].isin(parcels_gdf[f'hcad_pad{pad_len}']).sum()
        match_pct = 100 * matches / len(df_hcad)
        print(f"    Padding {pad_len:2d}: {matches:,} matches ({match_pct:.1f}%)")
        
        if matches > best_match:
            best_match = matches
            best_pad = pad_len
    
    print(f"\n  ✓ Using padding length {best_pad} ({best_match:,} matches expected)")
    
    # Do the actual merge
    df_hcad['acct_normalized'] = df_hcad['acct'].astype(str).str.zfill(best_pad)
    parcels_gdf['hcad_normalized'] = parcels_gdf['HCAD_NUM'].astype(str).str.zfill(best_pad)
    
    shp_cols = parcels_gdf[['hcad_normalized', 'latitude', 'longitude']].copy()
    
    df_hcad_joined = df_hcad.merge(
        shp_cols,
        left_on='acct_normalized',
        right_on='hcad_normalized',
        how='left',
        suffixes=('_hcad', '_shp')
    )
    
    matched_count = df_hcad_joined['latitude_shp'].notna().sum()
    match_pct = 100 * matched_count / len(df_hcad)
    
    print(f"\n  Merge result: {matched_count:,} / {len(df_hcad):,} matched ({match_pct:.1f}%)")
    
    if matched_count < len(df_hcad) * 0.8:
        print(f"  ⚠️  WARNING: Less than 80% matched!")
    else:
        print(f"  ✓ Match rate acceptable")
    
    # Use shapefile coordinates where available, fallback to HCAD otherwise
    df_hcad_joined['latitude'] = df_hcad_joined['latitude_shp'].fillna(df_hcad_joined['latitude_hcad'])
    df_hcad_joined['longitude'] = df_hcad_joined['longitude_shp'].fillna(df_hcad_joined['longitude_hcad'])
    
    # Keep only needed columns (note: zone_cat will be created in next cell)
    df_hcad = df_hcad_joined[[
        'acct', 'latitude', 'longitude', 
        'numfloors', 'yearbuilt', 'bldgarea', 'numbldgs', 'lotarea', 'landuse'
    ]].copy()
    
    print(f"\n  Final geocoded data:")
    print(f"    Records: {len(df_hcad):,}")
    print(f"    Lat: {df_hcad['latitude'].min():.4f} to {df_hcad['latitude'].max():.4f}")
    print(f"    Lon: {df_hcad['longitude'].min():.4f} to {df_hcad['longitude'].max():.4f}")
    print(f"    Unique lat/lon pairs: {df_hcad[['latitude', 'longitude']].drop_duplicates().shape[0]:,}")
    
except Exception as e:
    import traceback
    print(f"\n✗ ERROR during geocoding: {e}")
    traceback.print_exc()
    print(f"\n  Proceeding with placeholder coordinates (analysis may be invalid)")

In [ ]:
# Map HCAD landuse codes to zone types
def hcad_to_zone_type(code):
    """Map HCAD landuse codes to geographic zone types."""
    if pd.isna(code) or str(code).strip() in ["", "0", "    "]:
        return "Unknown"
    try:
        code_int = int(str(code).strip())
    except:
        return "Unknown"
    
    # HCAD landuse code ranges
    if 1000 <= code_int < 2000:
        return "Residential"       # 1001-1999: residential
    elif 2000 <= code_int < 4000:
        return "Commercial"        # 2000-3999: retail & office
    elif 4000 <= code_int < 7000:
        return "Industrial"        # 4000-6999: industrial & warehouse
    elif 7000 <= code_int < 8000:
        return "Infrastructure"    # 7000-7999: infrastructure
    elif 8000 <= code_int < 9000:
        return "Institutional"     # 8000-8999: public/institutional
    else:
        return "Other"             # 9000+: agricultural/exempt

df_hcad["zone_cat"] = df_hcad["landuse"].apply(hcad_to_zone_type)

print("\nZone type distribution:")
zone_counts = df_hcad["zone_cat"].value_counts().to_dict()
for zone, count in sorted(zone_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {zone:20s}: {count:>8,} ({100*count/len(df_hcad):>5.1f}%)")

In [ ]:
# Generate regular grid cells
def cell_size_degrees(cell_size_m, center_lat):
    """Convert cell size in meters to degrees of latitude and longitude."""
    R = 6.371e6  # Earth radius in meters
    lat_deg = cell_size_m / (R / 180 * np.pi)
    lon_deg = cell_size_m / (R * np.cos(np.radians(center_lat)) / 180 * np.pi)
    return lat_deg, lon_deg

# Get bounding box with padding
lat_min = df_hcad["latitude"].min() - 0.01
lat_max = df_hcad["latitude"].max() + 0.01
lon_min = df_hcad["longitude"].min() - 0.01
lon_max = df_hcad["longitude"].max() + 0.01
center_lat = (lat_min + lat_max) / 2

# Calculate grid spacing
lat_deg, lon_deg = cell_size_degrees(CELL_SIZE_M, center_lat)

print(f"\nGenerating grid:")
print(f"  Cell size: {CELL_SIZE_M}m")
print(f"  Grid spacing: {lat_deg:.6f}° lat × {lon_deg:.6f}° lon")
print(f"  Bounding box: ({lat_min:.4f}, {lon_min:.4f}) to ({lat_max:.4f}, {lon_max:.4f})")

# Create grid lines
lats = np.arange(lat_min, lat_max + lat_deg, lat_deg)
lons = np.arange(lon_min, lon_max + lon_deg, lon_deg)

print(f"  Grid dimensions: {len(lats)-1} rows × {len(lons)-1} cols = {(len(lats)-1)*(len(lons)-1):,} cells")

# Generate grid cells
grid_cells = []
for i in range(len(lats) - 1):
    for j in range(len(lons) - 1):
        grid_cells.append({
            "cell_id": f"cell_{len(grid_cells):08d}",
            "cell_lat": (lats[i] + lats[i+1]) / 2,
            "cell_lon": (lons[j] + lons[j+1]) / 2,
            "lat_min": lats[i],
            "lat_max": lats[i+1],
            "lon_min": lons[j],
            "lon_max": lons[j+1],
        })

df_grid = pd.DataFrame(grid_cells)
print(f"\n✓ Generated {len(df_grid):,} grid cells")

In [ ]:
# Assign HCAD records to nearest grid cell using BallTree
print(f"\nAssigning {len(df_hcad):,} properties to grid cells...")

coords = df_grid[["cell_lat", "cell_lon"]].values
tree = BallTree(np.radians(coords), metric="haversine")

hcad_coords = df_hcad[["latitude", "longitude"]].values
distances, indices = tree.query(np.radians(hcad_coords), k=1)

df_hcad["cell_id"] = df_grid.iloc[indices.flatten()]["cell_id"].values

# Check assignment distribution
assignment_counts = df_hcad["cell_id"].value_counts()
print(f"\n✓ Assigned {len(df_hcad):,} records to cells")
print(f"  Cells with data: {len(assignment_counts):,}")
print(f"  Properties per cell - Min: {assignment_counts.min()}, Max: {assignment_counts.max()}, Median: {assignment_counts.median():.0f}")
print(f"  Distance stats (km): Min: {distances.min()*6371:.3f}, Max: {distances.max()*6371:.3f}, Mean: {distances.mean()*6371:.3f}")

In [ ]:
# Aggregate by cell: compute area-weighted zone type
print(f"\nAggregating properties by cell...")
print(f"  Min lots per cell (filter): {MIN_LOTS}")

# Ensure lotarea is numeric
df_hcad["lotarea"] = pd.to_numeric(df_hcad["lotarea"], errors="coerce").fillna(1)

cell_summaries = []
cells_filtered = 0

for cell_id in df_grid["cell_id"]:
    cell_data = df_hcad[df_hcad["cell_id"] == cell_id]
    
    # Filter by minimum lot count
    if len(cell_data) < MIN_LOTS:
        cells_filtered += 1
        continue
    
    # Compute area-weighted zone type
    total_area = cell_data["lotarea"].sum()
    if total_area == 0:
        zone_type = "Unknown"
    else:
        zone_areas = cell_data.groupby("zone_cat")["lotarea"].sum()
        primary_zone = zone_areas.idxmax()
        primary_fraction = zone_areas[primary_zone] / total_area
        # Zone type is primary if ≥50% of area, else mixed-use
        zone_type = primary_zone if primary_fraction >= 0.5 else "Mixed-Use"
    
    # Get cell metadata
    cell_info = df_grid[df_grid["cell_id"] == cell_id].iloc[0]
    
    cell_summaries.append({
        "cell_id": cell_id,
        "cell_lat": cell_info["cell_lat"],
        "cell_lon": cell_info["cell_lon"],
        "zone_type": zone_type,
        "cell_lot_count": len(cell_data),
    })

df_output = pd.DataFrame(cell_summaries)

print(f"\n✓ Aggregation complete:")
print(f"  Grid cells total: {len(df_grid):,}")
print(f"  Cells filtered (< {MIN_LOTS} lots): {cells_filtered:,}")
print(f"  Cells output: {len(df_output):,}")
print(f"\nZone type distribution in output:")
zone_dist = df_output['zone_type'].value_counts().to_dict()
for zone, count in sorted(zone_dist.items(), key=lambda x: x[1], reverse=True):
    print(f"  {zone:20s}: {count:>6,}")

In [ ]:
# Save output
output_path = f"{CSV_DIR}/01_grid_definition.csv"

df_output[["cell_id", "cell_lat", "cell_lon", "zone_type", "cell_lot_count"]].to_csv(
    output_path, index=False
)

print(f"✓ Saved: {output_path}")
print(f"\nOutput summary:")
print(f"  Records: {len(df_output):,}")
print(f"  File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")
print(f"\nFirst 5 rows:")
print(df_output.head())
print(f"\nLast 5 rows:")
print(df_output.tail())